# 🚀 Experimental Benchmarking of Predicting Loan Payback Classification Models

This Jupyter Notebook serves as a research and experimental environment for the loan repayment prediction task.

## Objective
The main goal is to **compare the performance** and multiple classification models
on loan data to predict whether a loan will be paid back.
It serves as a benchmark for comparing the performance of different machine learning
algorithms on the same dataset.

## Benchmarked Algorithms
* **Random Forest**
* **XGBoost**
* **LightGBM**
* **CatBoost**

## Phases of the Experiment
1.  **Data Loading and Exploration:** Basic inspection of the dataset.
2.  **Preprocessing and Transformations:** Adapting the data (encoding, type handling) to meet the requirements of each model.
3.  **Training and Validation:** Training all four models on a unified training/validation split.
4.  **Results Analysis:** Comparing performance metrics (primarily **ROC-AUC**) and analyzing feature importance.

## Research Environment
Unlike production code, this notebook allows for iterative testing of different hyperparameters and preprocessing strategies for each model, focusing on rapid experimentation and deep analysis of the results.


In [1]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
print("numpy version",np.__version__)
print("pandas version",pd.__version__)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pickle
import gc


numpy version 1.26.4
pandas version 2.3.3


## 💾 Data Loading and Initial Analysis

In [2]:
data_path = os.path.join("..", "..", "data")
print("Zawartość folderu data/:", os.listdir(data_path))
data_train_path = os.path.join(data_path, "train.csv")
data_test_path = os.path.join(data_path, "test.csv")

print(data_train_path)
print(data_test_path)
try :
    df = pd.read_csv(data_train_path)
    df_test = pd.read_csv(data_test_path)
except:
    print("Zawartość folderu data/:", os.listdir(data_path))
    print(f"❌ Failed to read data from {data_train_path} or {data_test_path}")
    sys.exit(1)

df.set_index('id', inplace=True)
df_test.set_index('id', inplace=True)

print("--- Initial Data ---")
print(df.head())
print("--- DataFrame Information ---")
print(df.info())
print("\n--- Missing Value Count ---")
print(df.isnull().sum())

list_num_columns = ['annual_income','debt_to_income_ratio','credit_score','loan_amount','interest_rate ']

list_cat_columns = ['gender','marital_status','education_level','employment_status',
                     'loan_purpose','grade_subgrade']

print('Uniq values in categorical columns')
print(df[list_cat_columns].nunique())

Zawartość folderu data/: ['sample_submission.csv', 'test.csv', 'train.csv']
..\..\data\train.csv
..\..\data\test.csv
--- Initial Data ---
    annual_income  debt_to_income_ratio  credit_score  loan_amount  \
id                                                                   
0        29367.99                 0.084           736      2528.42   
1        22108.02                 0.166           636      4593.10   
2        49566.20                 0.097           694     17005.15   
3        46858.25                 0.065           533      4682.48   
4        25496.70                 0.053           665     12184.43   

    interest_rate  gender marital_status education_level employment_status  \
id                                                                           
0           13.67  Female         Single     High School     Self-employed   
1           12.92    Male        Married        Master's          Employed   
2            9.76    Male         Single     High School   

## Transformation functions

### Chenge in data frame columns from the list to "category" type

In [3]:
def cat_columns_to_category(df, list_cat_columns):

    for col in list_cat_columns:
        df[col] = df[col].astype('category')
    
    return df

### 🔪 Data Splitting
In machine learning workflows, datasets are typically divided into three key subsets:

1. **Training set** – used to train the model.
2. **Validation set** – used to tune hyperparameters and monitor model performance during training.
3. **Test set** – used at the final stage to evaluate how well the model generalizes to unseen data.

🎯 Why split the data?
- **Prevents overfitting** – the model learns general patterns instead of memorizing the data.
- **Ensures generalization** – we test how well the model performs on new, unseen samples.
- **Supports optimization** – the validation set helps select the best model configuration without compromising the integrity of the test set.

🔀 The data is split randomly into subsets, ensuring that samples are drawn from different parts of the dataset. This helps maintain diversity and representativeness across the training and validation sets. By doing so, the model is exposed to a broad range of examples during training and evaluation, which is especially important when dealing with uneven or complex data distributions.

In [4]:
def split_data(df, target_col='defaulted', test_size=0.2, random_state=42):
    from sklearn.model_selection import train_test_split
    import gc
        # --- DATA SPLIT: Train and Validation ---
    X_train_full_data = df.drop([target_col], axis=1)
    y_train_full_data = df[target_col]

    X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
        X_train_full_data, y_train_full_data,
        test_size=test_size,
        random_state=random_state,
        stratify=y_train_full_data
    )
    train_indices = X_train_split.index.values
    valid_indices = X_valid_split.index.values

    print(f"Total training data size: {len(df)}")
    print(f"New Train Set size: {len(X_train_split)}")
    print(f"Validation Set size: {len(X_valid_split)}")

    # Free memory
    del y_train_full_data
    gc.collect()
    return train_indices, valid_indices, X_train_split, X_valid_split, y_train_split, y_valid_split

### 🧮 One-Hot Encoding (OHE) 
One-hot encoding is a method used to **convert categorical columns into binary columns**. Each unique category becomes a separate column with **values 0 or 1**.  
- This function uses **pd.get_dummies() to perform one-hot encoding**.  
- The parameter **drop_first=True** removes the first category from each column to avoid the dummy variable trap (can cause redundant information) — this helps reduce the number of features and prevents multicollinearity during model training.  
- pd.get_dummies() **ignores NaN values** by default — it does not create a separate column for NaN.
#### ⚠️
Note: When drop_first=True is used and the column contains NaN values, rows with missing values may end up with **zeros in all one-hot columns**, which can lead to loss of information during training or inference.

In [5]:
def cat_One_hot_encoding(df,list_cat_columns,list_num_columns,train=False):
    import pickle
    import os
    # function transforms every categorical data column to binery multiple columns 
    # One-hot encoding
    df = pd.get_dummies(df,columns = list_cat_columns,drop_first=True)
    
    # Find all columns created by one-hot encoding (everything except numeric columns)
    one_hot_encoded_columns = df.columns.difference(list_num_columns)
    
    # Check if one-hot encoded columns contain only 0 or 1 values
    for col in one_hot_encoded_columns:
        if not df[col].dropna().apply(lambda x: x in [0, 1]).all():
            print(f"Non-binary values found in column: {col}")
            print(df[col].unique())

    # Convert one-hot encoded columns to integer type (0 or 1)
    df[one_hot_encoded_columns] = df[one_hot_encoded_columns].astype(int)

    # If training columns are provided, reindex to match them and fill missing columns with 0
    data_path = os.path.join("exp_train_columns.pkl")
    if train == False :
        with open(data_path , "rb") as f:
            train_columns = pickle.load(f)
        df = df.reindex(columns=train_columns, fill_value=0)
    else :
        print("💾 Saving train columns")
        with open(data_path , "wb") as f:
            pickle.dump(df.columns.tolist(), f)
    return df

## Model Training Functions

In [6]:
def train_model(model_type, X_train, y_train, X_valid, y_valid, params=None, results_df=None):
    """
    Trains a selected model and evaluates it using ROC AUC.
    """
    import time
    from datetime import datetime
    from sklearn.metrics import roc_auc_score
    import pandas as pd

    start_time = time.time()
    current_time = datetime.now().strftime("%H:%M:%S")

    # --- Model Selection ---
    if model_type == 'xgboost':
        import xgboost as xgb
        base_model = xgb.XGBClassifier()
    elif model_type == 'lightgbm':
        import lightgbm as lgb
        base_model = lgb.LGBMClassifier()
    elif model_type == 'random_forest':
        from sklearn.ensemble import RandomForestClassifier
        base_model = RandomForestClassifier()
    else:
        raise ValueError(f"Unsupported model_type: {model_type}")

    # --- Merge with default parameters ---
    if params is not None:
        default_params = base_model.get_params()
        default_params.update(params)
        base_model.set_params(**default_params)

    model = base_model

    # --- Training ---
    model.fit(X_train, y_train)

    # --- Evaluation ---
    p_valid = model.predict_proba(X_valid)[:, 1]
    auc_score = roc_auc_score(y_valid, p_valid)
    duration = (time.time() - start_time) / 60

    print(f"\n[{model_type.upper()}] ROC AUC: {auc_score:.5f} | Time: {duration:.2f} min")

    # --- Logging ---
    log_entry = {
        "model_name": model_type,
        "score": round(auc_score, 5),
        "duration_min": round(duration, 2),
        "timestamp": current_time,
        **model.get_params()
    }

    if results_df is None:
        results_df = pd.DataFrame([log_entry])
    else:
        results_df = pd.concat([results_df, pd.DataFrame([log_entry])], ignore_index=True)
    
    return model, model.get_params(), results_df

In [7]:
def train_catboost_baseline_auc(
    X_train_pool,
    X_valid_pool,
    # X_test_pool,
    y_valid_split,
    df_model,
    model_name="CatBoost_Base",
    custom_params=None
):
    """
    Trains a CatBoost model using default or user-defined parameters and evaluates performance using ROC AUC.

    Args:
        X_train_pool: CatBoost Pool for training
        X_valid_pool: CatBoost Pool for validation
        X_test_pool: CatBoost Pool for testing
        y_valid_split: True labels for validation set
        df_model: DataFrame to log model results
        model_name: Name to assign to the model
        custom_params: Optional dictionary of CatBoost hyperparameters

    Returns:
        model: trained CatBoost model
        used_params: final parameter dictionary
        df_model: updated log DataFrame
        p_test: predicted probabilities on test set
    """
    import time
    from datetime import datetime
    from sklearn.metrics import roc_auc_score
    from catboost import CatBoostClassifier
    import pandas as pd

    start_time = time.time()
    current_time = datetime.now().strftime("%H:%M:%S")
    print(f"\n--- Starting CatBoost Training at {current_time} ---")

    # Create base model with default parameters
    model = CatBoostClassifier()

    # Get default parameters from model
    used_params = model.get_params()

    # Update with any custom parameters provided
    if custom_params is not None:
        used_params.update(custom_params)

    # Re-initialize model with updated parameters
    model = CatBoostClassifier(**used_params)

    # Train model
    model.fit(
        X_train_pool,
        eval_set=X_valid_pool,
        verbose=used_params.get('verbose', 100),
        early_stopping_rounds=used_params.get('early_stopping_rounds', None)
    )

    # Evaluate on validation set
    p_valid = model.predict_proba(X_valid_pool)[:, 1]
    auc_score = roc_auc_score(y_valid_split, p_valid)
    print(f"\nValidation ROC AUC Score: {auc_score:.5f}")

    # Predict on test set
    # p_test = model.predict_proba(X_test_pool)[:, 1]
    duration = (time.time() - start_time) / 60
    print(f"Model Training Time: {duration:.2f} minutes")

    # Log results
    log_entry = {
        "model_name": model_name,
        "metric": "ROC AUC",
        "score": round(auc_score, 5),
        "duration_min": round(duration, 2),
        "timestamp": current_time,
        **used_params
    }

    df_model = pd.concat([
        df_model,
        pd.DataFrame([log_entry])
    ], ignore_index=True)

    return model, used_params, df_model

In [8]:
def tune_catboost_classifier_auc(X_train_pool, X_valid_pool, y_valid, tuning_results_df=None, tuning_log_df=None):
    """
    Performs a Grid Search for the CatBoost Classifier using ROC AUC as the evaluation metric.
    Returns:
        - Best trained CatBoost model
        - Best parameter dictionary
        - Updated DataFrame with best models only
        - Full log DataFrame with all tested configurations
    """
    import time
    from datetime import datetime
    from catboost import CatBoostClassifier
    from sklearn.metrics import roc_auc_score, log_loss
    import pandas as pd
    import gc

    # --- Hyperparameter Grid ---
    search_iterations = [3000]
    search_learning_rates = [0.06,0.07,0.075]
    search_depths = [ 8,9,10]
    search_l2_leaf_regs = [6, 10]
    search_bagging_temperatures = [1]
    search_border_counts = [254]

    # --- Imbalance Handling ---
    positive_count = y_valid.sum()
    negative_count = len(y_valid) - positive_count
    scale_pos_weight = int(negative_count / positive_count)

    # --- Tracking Setup ---
    best_auc_roc = 0
    best_params = {}
    best_model = None
    counter = 0
    start_time = time.time()

    # --- Initialize DataFrames if not provided ---
    if tuning_results_df is None:
        tuning_results_df = pd.DataFrame()

    if tuning_log_df is None:
        tuning_log_df = pd.DataFrame()

    # --- Grid Search Loop ---
    for iterations_ in search_iterations:
        for lr in search_learning_rates:
            for depth in search_depths:
                depth_start_time = time.time()
                for l2_leaf_reg_ in search_l2_leaf_regs:
                    for bagging_temperature_ in search_bagging_temperatures:
                        for border_count in search_border_counts:
                            counter += 1
                            print(f"\n--- Model {counter} --- Testing LR: {lr}, Depth: {depth}, L2: {l2_leaf_reg_}")
                            start_time_l = time.time()

                            cat_params = {
                                'loss_function': 'Logloss',
                                'eval_metric': 'CrossEntropy',
                                'scale_pos_weight': scale_pos_weight,
                                'learning_rate': lr,
                                'iterations': iterations_,
                                'depth': depth,
                                'l2_leaf_reg': l2_leaf_reg_,
                                'bagging_temperature': bagging_temperature_,
                                'border_count': border_count,
                                'task_type': 'GPU',
                                'random_seed': 42,
                                'verbose': 0,
                                'early_stopping_rounds': 200
                            }

                            model = CatBoostClassifier(**cat_params)
                            model.fit(X_train_pool, eval_set=X_valid_pool)

                            p_valid = model.predict_proba(X_valid_pool)[:, 1]
                            auc_roc = roc_auc_score(y_valid, p_valid)
                            logloss = log_loss(y_valid, p_valid)
                            duration = (time.time() - start_time_l) / 60
                            current_time = datetime.now().strftime("%H:%M:%S")

                            log_entry = {
                                "model_index": counter,
                                "score_auc_roc": round(auc_roc, 5),
                                "score_logloss": round(logloss, 5),
                                "duration_min": round(duration, 2),
                                "timestamp": current_time,
                                **cat_params
                            }

                            # Zapisz pełny log każdej konfiguracji
                            tuning_log_df = pd.concat([
                                tuning_log_df,
                                pd.DataFrame([log_entry])
                            ], ignore_index=True)

                            # Zapisz tylko najlepsze modele do głównego rejestru
                            if auc_roc > best_auc_roc:
                                best_auc_roc = auc_roc
                                best_params = cat_params
                                best_model = model
                                print(f"---> NEW BEST ROC AUC: {best_auc_roc:.5f}")
                                print(f"     Best Parameters: {best_params}")

                                best_entry = {
                                    "model_name": "catboost_tune",
                                    "metric": "ROC AUC",
                                    "score": round(auc_roc, 5),
                                    "duration_min": round(duration, 2),
                                    "timestamp": current_time,
                                    **cat_params
                                }

                                tuning_results_df = pd.concat([
                                    tuning_results_df,
                                    pd.DataFrame([best_entry])
                                ], ignore_index=True)

                            del model
                            gc.collect()

                depth_end_time = time.time()
                print(f"Time for Depth {depth} sweep: {(depth_end_time - depth_start_time)/60:.2f} min")

    total_time = (time.time() - start_time) / 60
    print(f"\n--- TUNING FINISHED --- Total execution time: {total_time:.2f} minutes")

    return best_model, best_params, tuning_results_df, tuning_log_df

# 🛠️ DATA TRANSFORMATIONS

In [9]:
# Transform categorical columns to 'category' dtype
df_cat = cat_columns_to_category(df, list_cat_columns)

# Split data into training and validation sets
train_indices, valid_indices, X_train_split, X_valid_split, y_train_split, y_valid_split= split_data(df_cat, target_col='loan_paid_back')

# One-Hot Encoding for categorical variables
X_train_ohe = cat_One_hot_encoding(X_train_split,list_cat_columns,list_num_columns,train=True)
X_valid_ohe = cat_One_hot_encoding(X_valid_split,list_cat_columns,list_num_columns,train=False)

X_train_ohe = X_train_ohe.fillna(0)
X_valid_ohe = X_valid_ohe.fillna(0)

X_valid_ohe.head()

Total training data size: 593994
New Train Set size: 475195
Validation Set size: 118799
Non-binary values found in column: interest_rate
[13.78 14.4  11.99 ...  6.95 19.08  5.42]
💾 Saving train columns
Non-binary values found in column: interest_rate
[11.8   7.46 12.73 ... 18.66  3.2   7.22]


,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender_Male,gender_Other,marital_status_Married,marital_status_Single,marital_status_Widowed,...,grade_subgrade_E1,grade_subgrade_E2,grade_subgrade_E3,grade_subgrade_E4,grade_subgrade_E5,grade_subgrade_F1,grade_subgrade_F2,grade_subgrade_F3,grade_subgrade_F4,grade_subgrade_F5
id,,,,,,,,,,,,,,,,,,,,,
523771,28114.66,0.080,690,7133.85,11,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
524810,122933.72,0.046,659,7268.12,7,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
100141,26379.82,0.140,710,9386.27,12,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
535634,15371.00,0.063,691,17053.01,9,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
518524,43295.36,0.130,756,16359.94,11,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


### 📐Column Alignment Check (Post-OHE)
After **One-Hot Encoding (OHE)**, it's critical to ensure that the **training**, **validation**, and **test** datasets all have the **exact same columns** in the **exact same order**.

This is done by re-indexing the validation and test sets to match the columns of the training set. If a category (a column) exists in the training data but not in the test data (or vice versa), the missing column is added and filled with **zeros** (`fill_value=0`).

#### ⚠️ A Crucial Step
Mismatching columns (feature sets) Error avoided. This check confirms that all three sets are **aligned and ready** for the final model training and prediction.

In [10]:
print(f"nr of features X_train_ohe {X_train_ohe.shape[1]}")
print(f"nr of features X_valid_ohe {X_valid_ohe.shape[1]}")

print(f"Count NaN in X_train_ohe : {X_train_ohe.isna().sum().sum()}")
print(f"Count NaN in X_valid_ohe : {X_valid_ohe.isna().sum().sum()}")

if X_train_ohe.columns.equals(X_valid_ohe.columns):
    print(f"columns count {len(X_valid_ohe.columns)}\nX_valid_ohe columns are aligned and ready for modeling. 🎉 ")
else:
    print('ERROR: Column mismatch detected! Check OHE and reindex logic.')

nr of features X_train_ohe 54
nr of features X_valid_ohe 54
Count NaN in X_train_ohe : 0
Count NaN in X_valid_ohe : 0
columns count 54
X_valid_ohe columns are aligned and ready for modeling. 🎉 


# 📖🔬 MODEL TRAINING

### results_df to log model results

In [11]:
results_df = pd.DataFrame()

## RandomForestClassifier

### 🧠 Data Prep Summary
Random Forest is a tree-based ensemble method that builds multiple decision trees and averages their predictions. 
Unlike boosting methods, it trains trees independently and is robust to overfitting and noise.

* **Categorical Features - One-Hot Encoding (OHE)** Features Must be encoded — Random Forest doesn't support them natively. 

* **Scaling Not needed** — tree models are scale-invariant. Raw numerical values work fine.

* **Missing Values Not handled automatically** — you need to impute missing values before training. 
Use SimpleImputer or similar preprocessing.

In [ ]:
model_RFC, model_params_RFC, results_df=train_model("random_forest", X_train_ohe, y_train_split, X_valid_ohe, y_valid_split, params=None, results_df=results_df)

## XGBoost Classifier

### 🧠 Data Prep Summary
XGBoost is a tree-based ensemble method using gradient boosting — unlike single decision trees, 
it builds many trees sequentially to correct previous errors and improve accuracy.

* **Categorical Features - One-Hot Encoding (OHE)** Features Must be encoded — XGBoost doesn’t support them natively.

* **Scaling Not needed** — tree models are scale-invariant. Raw numerical values work fine.

* **Missing Values Handled internally** (np.nan is supported). No manual imputation required, 
but it's good to monitor missing data.

In [ ]:
model_XGB, model_params_XGB, results_df=train_model("xgboost", X_train_ohe, y_train_split, X_valid_ohe, y_valid_split, params=None, results_df=results_df)

## LGBMClassifier

### 🧠 Data Prep Summary
LightGBM is a tree-based boosting algorithm optimized for speed and memory efficiency. Unlike traditional decision trees, it builds trees leaf-wise for better accuracy and supports large datasets with high performance.

* **Categorical Features — no need for one-hot encoding** Features Can be passed directly as category dtype. LightGBM handles categorical splits natively.

* **Scaling Not needed** — tree models are scale-invariant. Raw numerical values work fine.

* **Missing Values Handled internally** (np.nan is supported). No manual imputation required, but it's good to monitor missing data.

In [ ]:
model_LGBM, model_params_LGBM, results_df=train_model("lightgbm", X_train_split, y_train_split, X_valid_split, y_valid_split, params=None, results_df= results_df)

## CatBoostClassifier

### 🧠 Data Prep & Modeling Summary
CatBoost is a gradient boosting algorithm based on decision trees, designed for high performance and native support for 
categorical features. Unlike other boosting methods, it handles categorical data internally and requires minimal preprocessing.

* **Categorical Features - No need for one-hot encoding** — CatBoost handles them natively.

* **Scaling Not needed**  — tree-based models are scale-invariant.

* **Missing Values Handled internally**  (np.nan is supported), but monitoring is recommended.

⚙️ Base Model Configuration & Imbalance Handling
Cost-Sensitive Learning scale_pos_weight was set as the ratio of negative to positive samples to penalize misclassification 
of the minority class(positive response), and improve AUC.
Acceleration & Evaluation Strategy
The parameter task_type='GPU' was enabled to leverage GPU acceleration, significantly reducing training time.
Since CatBoost does not support AUC as a loss function, cross-entropy (logloss) was used for training and early stopping.
AUC was calculated only after training, on the validation set, to evaluate model performance and guide final selection.
Regularization & Control High iterations (e.g. 15000) combined with early_stopping_rounds=200 helped prevent overfitting and 
identify the optimal number of trees.


In [ ]:
# Creating CatBoost Data Pools
from catboost import CatBoostClassifier, Pool
X_train_pool = Pool(X_train_split, y_train_split, cat_features=list_cat_columns)
X_valid_pool = Pool(X_valid_split, y_valid_split, cat_features=list_cat_columns)

custom_params = {
    'eval_metric': 'AUC',
    'learning_rate': 0.07,
    'iterations': 5000,
    'early_stopping_rounds': 200
}

cat_clf, cat_params, results_df = train_catboost_baseline_auc(
    X_train_pool, X_valid_pool, y_valid_split,
    df_model=results_df,
    model_name="CatBoost_baseline",
    custom_params=custom_params
)

In [ ]:
cat_tune_model, best_params_cat_tune, df_model_results, tuning_log_df = tune_catboost_classifier_auc(
    X_train_pool, X_valid_pool, y_valid, df_model_results)

In [ ]:
import matplotlib.pyplot as plt

def plot_metric(df, metric_name):
    # Sortujemy po wyniku, ale zachowujemy model_index jako etykietę
    df_sorted = df.sort_values(by=metric_name)
    df_sorted = df_sorted.reset_index(drop=True)
    df_sorted['rank'] = range(1, len(df_sorted) + 1)  # Dodajemy numer porządkowy

    plt.figure(figsize=(10, 5))
    plt.plot(df_sorted['rank'], df_sorted[metric_name], marker='o')
    plt.xticks(df_sorted['rank'], df_sorted['model_index'], rotation=45)
    plt.title(f'{metric_name} vs Model Index')
    plt.xlabel(f'Model Index (sorted by {metric_name})')
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric(tuning_log_df, 'score_auc_roc')
plot_metric(tuning_log_df, 'score_logloss')

# 🥇 Model Training Summary

In [ ]:
print("\n=== Model Training Summary ===")
print(results_df)
# Save the best model (based on ROC AUC)
best_model_row = results_df.loc[results_df['score'].idxmax()]
best_model_name = best_model_row['model_name']
print(f"\nBest Model: {best_model_name} with ROC AUC: {best_model_row['score']}")

# # Ścieżka do zapisu
# model_path = os.path.join(os.path.dirname(__file__),"app","model.pkl")

# # Zapis modelu
# with open(model_path, "wb") as f:
#     pickle.dump(cat_clf, f)

# print(f"💾 Model zapisany do: {model_path}")

In [ ]:
def plot_feature_importance(model, feature_dataframe, top_n=10, plot_title="Model Feature Importance"):
    """
    Retrieves, sorts, and visualizes the feature importance from a trained CatBoost model.

    Args:
        model (CatBoostClassifier): The trained CatBoost model object.
        feature_dataframe (pd.DataFrame): The DataFrame containing the features used for training.
        top_n (int): The number of top features to display.
        plot_title (str): The title for the resulting plot.

    Returns:
        pd.DataFrame: A DataFrame of sorted feature importances.
    """
    
    if not hasattr(model, 'get_feature_importance'):
        print("Error: The provided object does not have a 'get_feature_importance' method.")
        return pd.DataFrame()

    try:
        # 1. Get Feature Importances
        feature_importances = model.get_feature_importance()
        feature_names = feature_dataframe.columns.tolist()

        # 2. Create and Sort DataFrame
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': feature_importances
        })
        importance_df = importance_df.sort_values(by='Importance', ascending=False)
        
        # Select top N features
        top_features = importance_df.head(top_n)

        print(f"\n--- TOP {top_n} {plot_title} ---\n")
        print(top_features)

        # 3. Visualization
        plt.figure(figsize=(10, top_n / 2)) # Dynamic height based on N
        plt.barh(top_features['Feature'], top_features['Importance'], color='teal')
        plt.xlabel("Feature Importance Value")
        plt.ylabel("Feature")
        plt.title(plot_title)
        plt.gca().invert_yaxis() # Invert y-axis for better readability
        
        # 💾 Save the plot

        image_path = os.path.join("images", "Feature_Importance.png")
        plt.savefig(image_path, bbox_inches='tight')
        
        plt.show() # 

        return importance_df

    except Exception as e:
        print(f"An error occurred during feature importance processing: {e}")
        return pd.DataFrame()
    

In [ ]:
loan_importance_df = plot_feature_importance(
    model=cat_clf,
    feature_dataframe=X_train_split,
    plot_title="Loan Risk Estamation Feature Importance"
)

In [12]:
# TEST

In [13]:
data_path = os.path.join("..", "..", "data")
print("Zawartość folderu data/:", os.listdir(data_path))
data_test_path = os.path.join(data_path, "test.csv")
data_sample_submission_path = os.path.join(data_path, "sample_submission.csv")

print('data_test_path ',data_test_path)

try :
    df_test = pd.read_csv(data_test_path)
    df_sample_submission = pd.read_csv(data_sample_submission_path)
except:
    print("Zawartość folderu data/:", os.listdir(data_path))
    print(f"❌ Failed to read data from {data_test_path}")
    sys.exit(1)

df_test.set_index('id', inplace=True)
print(df_sample_submission.head())
print(len(df_test))

Zawartość folderu data/: ['sample_submission.csv', 'test.csv', 'train.csv']
data_test_path  ..\..\data\test.csv
       id  loan_paid_back
0  593994               0
1  593995               0
2  593996               0
3  593997               0
4  593998               0
254569


In [17]:
# Creating CatBoost Data Pools
from catboost import Pool
model_path = os.path.join("model.pkl")
print("model_path -> ",model_path)
with open(model_path , "rb") as f:
    model = pickle.load(f)

df_test_cat = cat_columns_to_category(df_test, list_cat_columns)
df_test_ohe = cat_One_hot_encoding(df_test_cat,list_cat_columns,list_num_columns,train=False)
df_test_ohe = df_test_ohe.fillna(0)

print(f"nr of features X_train_ohe {X_train_ohe.shape[1]}")
print(f"nr of features df_test_ohe {df_test_ohe.shape[1]}")
print(f"Count NaN in X_train_ohe : {X_train_ohe.isna().sum().sum()}")
print(f"Count NaN in df_test_ohe : {df_test_ohe.isna().sum().sum()}")
if X_train_ohe.columns.equals(df_test_ohe.columns):
    print(f"columns count {len(df_test_ohe.columns)}\ndf_test_ohe columns are aligned and ready for modeling. 🎉 ")
else:
    print('ERROR: Column mismatch detected! Check OHE and reindex logic.')

X_test_pool  = Pool(df_test_cat, cat_features=list_cat_columns)


model_path ->  model.pkl
Non-binary values found in column: interest_rate
[14.73 12.85 13.29 ...  3.97 17.47  5.4 ]
nr of features X_train_ohe 54
nr of features df_test_ohe 54
Count NaN in X_train_ohe : 0
Count NaN in df_test_ohe : 0
columns count 54
df_test_ohe columns are aligned and ready for modeling. 🎉 


In [18]:
probability = model.predict_proba(X_test_pool)
#risk = "low" if probability < 0.5 else "high"
print(probability)


[[0.05034235 0.94965765]
 [0.01935092 0.98064908]
 [0.57312394 0.42687606]
 ...
 [0.03425113 0.96574887]
 [0.01473849 0.98526151]
 [0.08744306 0.91255694]]


In [19]:
import pandas as pd
print(len(df_test))
print(len(probability[:, 1]))
submission = pd.DataFrame({
    "id": df_test.index,
    "loan_paid_back": probability[:, 1] 
})
submission.to_csv("submission.csv", index=False)

254569
254569
